# 第二十八课:GRPO——组内基线的强化学习

讲义:[28-grpo.md](28-grpo.md)。规则奖励 + 组内基线 + PPO 裁剪 + KL 皮筋;训练篇终章四段对照(base → SFT → DPO → GRPO)。

In [ ]:
import math

import time

import numpy as np



# ===== 第 20 课 MiniGPT 原码(含 generate) =====

def softmax(x, axis=-1):

    m = np.max(x, axis=axis, keepdims=True)

    e = np.exp(x - m)

    return e / e.sum(axis=axis, keepdims=True)



class MiniGPT:

    def __init__(self, vocab=16, d=32, n_heads=4, n_layers=2, seed=0):

        rng = np.random.default_rng(seed)

        s = 0.1

        self.E = rng.normal(size=(vocab, d)) * 0.5

        self.dh = d // n_heads

        self.H, self.D, self.NL = n_heads, d, n_layers

        self.layers = [{

            'Wq': rng.normal(size=(d, d))*s, 'Wk': rng.normal(size=(d, d))*s,

            'Wv': rng.normal(size=(d, d))*s, 'Wo': rng.normal(size=(d, d))*s,

            'W1': rng.normal(size=(d, 4*d))*s*0.8, 'W2': rng.normal(size=(4*d, d))*s*0.8,

            'W3': rng.normal(size=(d, 4*d))*s*0.8,

        } for _ in range(n_layers)]

        self.head = rng.normal(size=(d, vocab)) * s



    def rms(self, x, eps=1e-5):

        return x / np.sqrt((x**2).mean(-1, keepdims=True) + eps)



    def rope(self, x, pos):

        half = x.shape[-1] // 2

        ang = pos[:, None] / (10000 ** (np.arange(half)*2/x.shape[-1]))

        c, s = np.cos(ang), np.sin(ang)

        x1, x2 = x[..., :half], x[..., half:]

        return np.concatenate([x1*c - x2*s, x1*s + x2*c], -1)



    def _attn(self, x, pos, W, cache):

        n = x.shape[0]

        q, k, v = self.rope(x@W['Wq'], pos), self.rope(x@W['Wk'], pos), x@W['Wv']

        if cache is not None:

            k, v = np.concatenate([cache[0], k]), np.concatenate([cache[1], v])

        T = k.shape[0]

        mask = np.triu(np.ones((n, T), bool), T - n + 1)

        outs = []

        for h in range(self.H):

            sl = slice(h*self.dh, (h+1)*self.dh)

            s = q[:, sl] @ k[:, sl].T / math.sqrt(self.dh)

            s = np.where(mask, -np.inf, s)

            outs.append(softmax(s) @ v[:, sl])

        return np.concatenate(outs, -1) @ W['Wo'], (k, v)



    def _ffn(self, x, W):

        h = x @ W['W1']

        return (h / (1 + np.exp(-h)) * (x @ W['W3'])) @ W['W2']



    def forward(self, ids, cache=None):

        pos0 = cache[0][0].shape[0] if cache else 0

        x = self.E[list(ids)]

        new_cache = []

        for i, W in enumerate(self.layers):

            a, kv = self._attn(x, np.arange(pos0, pos0+len(ids)), W,

                               cache[i] if cache else None)

            x = x + a

            x = x + self._ffn(self.rms(x), W)

            new_cache.append(kv)

        logits = self.rms(x) @ self.head

        assert np.isfinite(logits).all(), 'NaN! 查掩码/精度/ε'

        return logits, new_cache



    def generate(self, ids, n, T=1.0, top_p=1.0, rng=None):

        rng = rng or np.random.default_rng()

        logits, cache = self.forward(ids)

        out = []

        for _ in range(n):

            p = softmax(logits[-1] / T)

            if top_p < 1.0:

                order = np.argsort(-p); cum = np.cumsum(p[order])

                keep = order[:np.searchsorted(cum, top_p)+1]

                m = np.zeros_like(p); m[keep] = p[keep]; p = m/m.sum()

            nxt = int(rng.choice(len(p), p=p/p.sum()))

            out.append(nxt)

            logits, cache = self.forward([nxt], cache)

        return list(ids) + out

In [ ]:
# ===== 第 23 课反向 + 本课 mask 扩展(合并为一个函数;mask=None 即 23 课原版) =====

def rms_backward(dy, x, eps=1e-5):

    r = np.sqrt((x**2).mean(-1, keepdims=True) + eps)

    y = x / r

    return (dy - y * (dy * y).mean(-1, keepdims=True)) / r



def rope_backward(dy, pos, dim):

    half = dim // 2

    ang = pos[:, None] / (10000 ** (np.arange(half)*2/dim))

    c, s = np.cos(ang), np.sin(ang)

    dy1, dy2 = dy[..., :half], dy[..., half:]

    return np.concatenate([dy1*c + dy2*s, -dy1*s + dy2*c], -1)



def silu_grad(u):

    sig = 1 / (1 + np.exp(-u))

    return sig * (1 + u * (1 - sig))



def loss_and_grads(gpt, ids, targets, mask=None):

    """23 课整机反向 + 25 课可选 mask(监督位加权)。mask=None 等价 23 课原版。"""

    T = len(ids)

    if mask is None:

        mask = [1.0] * T

    marr = np.array(mask, dtype=float)

    Ns = marr.sum()

    pos = np.arange(T)

    tape = []

    x = gpt.E[list(ids)]

    for W in gpt.layers:

        q, k, v = gpt.rope(x@W['Wq'], pos), gpt.rope(x@W['Wk'], pos), x@W['Wv']

        m = np.triu(np.ones((T, T), bool), 1)

        ps, outs = [], []

        for h in range(gpt.H):

            sl = slice(h*gpt.dh, (h+1)*gpt.dh)

            s = q[:, sl] @ k[:, sl].T / math.sqrt(gpt.dh)

            s = np.where(m, -np.inf, s)

            p = softmax(s)

            ps.append(p); outs.append(p @ v[:, sl])

        co = np.concatenate(outs, -1)

        xm = x + co @ W['Wo']

        xr = gpt.rms(xm)

        u, zv = xr @ W['W1'], xr @ W['W3']

        gate = u/(1+np.exp(-u))*zv

        tape.append(dict(x_in=x, xm=xm, xr=xr, u=u, zv=zv, gate=gate, co=co, q=q, k=k, v=v, ps=ps))

        x = xm + gate @ W['W2']                 # 注意:x 入 tape 在覆盖之前(25 课战争故事②)



    xf = x

    logits = gpt.rms(xf) @ gpt.head

    z = logits - logits.max(-1, keepdims=True)

    logp = z - np.log(np.exp(z).sum(-1, keepdims=True))

    loss = -(logp[np.arange(T), np.array(targets)] * marr).sum() / Ns



    grads = {'E': np.zeros_like(gpt.E), 'head': np.zeros_like(gpt.head),

             'layers': [dict.fromkeys(W) for W in gpt.layers]}

    p = softmax(logits)

    onehot = np.zeros_like(p)

    onehot[np.arange(T), np.array(targets)] = 1.0

    dlogits = (p - onehot) * (marr / Ns)[:, None]      # ← mask 版与 23 课的核心差别

    grads['head'] = gpt.rms(xf).T @ dlogits

    dx = rms_backward(dlogits @ gpt.head.T, xf)

    for i in reversed(range(gpt.NL)):

        W, t = gpt.layers[i], tape[i]

        dg = dx @ W['W2'].T

        du = dg * silu_grad(t['u']) * t['zv']

        dzv = dg * (t['u']/(1+np.exp(-t['u'])))

        grads['layers'][i]['W2'] = t['gate'].T @ dx

        grads['layers'][i]['W1'] = t['xr'].T @ du

        grads['layers'][i]['W3'] = t['xr'].T @ dzv

        dxm = dx + rms_backward(du @ W['W1'].T + dzv @ W['W3'].T, t['xm'])

        dc = dxm @ W['Wo'].T

        grads['layers'][i]['Wo'] = t['co'].T @ dxm

        dvs, dqs, dks = [], [], []

        for h in range(gpt.H):

            sl = slice(h*gpt.dh, (h+1)*gpt.dh)

            p_h, v_h, q_h, k_h = t['ps'][h], t['v'][:, sl], t['q'][:, sl], t['k'][:, sl]

            dO = dc[:, sl]

            dvs.append(p_h.T @ dO)

            dp = dO @ v_h.T

            ds = p_h * (dp - (dp*p_h).sum(-1, keepdims=True))

            dqs.append(ds @ k_h / math.sqrt(gpt.dh))

            dks.append(ds.T @ q_h / math.sqrt(gpt.dh))

        dv = np.concatenate(dvs, -1)

        dq = rope_backward(np.concatenate(dqs, -1), pos, gpt.D)

        dk = rope_backward(np.concatenate(dks, -1), pos, gpt.D)

        grads['layers'][i]['Wv'] = t['x_in'].T @ dv

        grads['layers'][i]['Wq'] = t['x_in'].T @ dq

        grads['layers'][i]['Wk'] = t['x_in'].T @ dk

        dx = dxm + dq @ W['Wq'].T + dk @ W['Wk'].T + dv @ W['Wv'].T

    np.add.at(grads['E'], np.array(ids), dx)

    return loss, grads



def flat_params(gpt):

    ps = [gpt.E, gpt.head]

    for W in gpt.layers:

        ps += [W[k] for k in ('Wq', 'Wk', 'Wv', 'Wo', 'W1', 'W2', 'W3')]

    return ps



def flat_grads(grads):

    gs = [grads['E'], grads['head']]

    for L in grads['layers']:

        gs += [L[k] for k in ('Wq', 'Wk', 'Wv', 'Wo', 'W1', 'W2', 'W3')]

    return gs



def clip_norms(gs, max_norm=1.0):

    total = math.sqrt(sum((g ** 2).sum() for g in gs))

    if total > max_norm:

        s = max_norm / total

        for g in gs:

            g *= s

    return total



class MultiAdamW:

    def __init__(self, params, lr=1e-3, betas=(0.9, 0.999), eps=1e-8, wd=0.01):

        self.ps, self.lr, self.b1, self.b2, self.eps, self.wd = params, lr, *betas, eps, wd

        self.m = [np.zeros_like(p) for p in params]

        self.v = [np.zeros_like(p) for p in params]

        self.t = 0

    def step(self, gs):

        self.t += 1

        for p, g, m, v in zip(self.ps, gs, self.m, self.v):

            m *= self.b1; m += (1 - self.b1) * g

            v *= self.b2; v += (1 - self.b2) * g * g

            p -= self.lr * (m/(1-self.b1**self.t)/(np.sqrt(v/(1-self.b2**self.t))+self.eps) + self.wd*p)

def lr_at(step, total, base_lr=1e-3, warmup=100, floor_ratio=0.1):
    if step < warmup:
        return base_lr * (step + 1) / warmup
    p = (step - warmup) / max(1, total - warmup)
    return base_lr * (floor_ratio + (1 - floor_ratio) * 0.5 * (1 + math.cos(math.pi * p)))

In [ ]:
# ===== 数据 + 模板 + 任务(讲义 §3、§5) =====

text = open("data/xiyouji.txt", encoding="utf-8").read()

chars = sorted(set(text))

stoi = {c: i for i, c in enumerate(chars)}

itos = {i: c for c, i in stoi.items()}

V0 = len(chars)

data = np.array([stoi[c] for c in text], dtype=np.int32)

n_train = int(len(data) * 0.99)

train_data, val_data = data[:n_train], data[n_train:]



T_USER, T_ASSI, T_END = V0, V0 + 1, V0 + 2



def enc(s):

    return [stoi[c] for c in s]



def extend_vocab(m, seed=1):

    r = np.random.default_rng(seed)

    m.E = np.vstack([m.E, r.normal(size=(3, m.D)) * 0.1])

    m.head = np.hstack([m.head, r.normal(size=(m.D, 3)) * 0.1])



ANTI = [("大","小"),("上","下"),("天","地"),("古","今"),("来","去"),("内","外"),

        ("高","低"),("长","短"),("黑","白"),("好","坏"),("真","假"),("生","死"),

        ("冷","热"),("快","慢"),("强","弱"),("轻","重"),("多","少"),("有","无"),

        ("明","暗"),("前","后"),("左","右"),("日","夜"),("开","关"),("爱","恨"),

        ("笑","哭"),("进","退"),("买","卖"),("胜","败"),("深","浅"),("忙","闲")]



COPY_STRINGS = ["金箍", "行者", "云端", "宝殿", "妖魔", "蒲团", "袈裟", "铁棒",

                "猴王", "灵山", "仙丹", "蟠桃", "佛祖", "天河", "石猴", "宝象",

                "法术", "神通", "筋斗", "火眼"]



def make_anti_ex(pair):

    q, a = pair

    ids = [T_USER] + enc(f"反义词:{q}") + [T_ASSI] + enc(a) + [T_END]

    mask = [0] * len(ids); mask[-2] = 1; mask[-1] = 1

    return ids, mask



def make_copy_ex(s):

    ids = [T_USER] + enc(f"重复:{s}") + [T_ASSI] + enc(s) + [T_END]

    mask = [0] * len(ids)

    for j in range(-len(s)-1, 0):

        mask[j] = 1

    return ids, mask



ALL_TASKS = [make_anti_ex(p) for p in ANTI] + [make_copy_ex(s) for s in COPY_STRINGS]

SPECIAL = {T_USER: '<|user|>', T_ASSI: '<|assistant|>', T_END: '<|end|>'}

def render(ids):

    return ''.join(SPECIAL[i] if i in SPECIAL else itos[i] for i in ids)

print(f"任务样本:{len(ANTI)} 反义词 + {len(COPY_STRINGS)} 复制 = {len(ALL_TASKS)}")

print("示例:", render(ALL_TASKS[0][0]), "| mask:", ALL_TASKS[0][1])

def val_loss_no_special(m):
    """全词表 val loss 与『去掉 3 个特殊 token 后在原词表上重归一化』的 val loss。"""
    rv = np.random.default_rng(7)
    off = rv.integers(0, len(val_data) - 65, 16)
    tot_full = tot_no = 0.0
    for o in off:
        xb = list(val_data[o:o+65])
        logits, _ = m.forward(xb[:-1])
        tgt = np.array(xb[1:])
        z = logits - logits.max(-1, keepdims=True)
        logp = z - np.log(np.exp(z).sum(-1, keepdims=True))
        tot_full += -logp[np.arange(len(tgt)), tgt].mean()
        z2 = logits[:, :V0] - logits[:, :V0].max(-1, keepdims=True)
        logp2 = z2 - np.log(np.exp(z2).sum(-1, keepdims=True))
        tot_no += -logp2[np.arange(len(tgt)), tgt].mean()
    return tot_full / 16, tot_no / 16

In [ ]:
# ===== 权重泛化:mask → 逐 token 权重(讲义 §3) =====

def weighted_loss_and_grads(gpt, ids, targets, weights):

    """dlogits = (p−onehot)·w_t,即 loss=−Σ w·logp 的梯度。CE 是 w=mask/N 的特例。"""

    T = len(ids)

    w = np.array(weights, dtype=float)

    pos = np.arange(T)

    tape = []

    x = gpt.E[list(ids)]

    for W in gpt.layers:

        q, k, v = gpt.rope(x@W['Wq'], pos), gpt.rope(x@W['Wk'], pos), x@W['Wv']

        m = np.triu(np.ones((T, T), bool), 1)

        ps, outs = [], []

        for h in range(gpt.H):

            sl = slice(h*gpt.dh, (h+1)*gpt.dh)

            s = q[:, sl] @ k[:, sl].T / math.sqrt(gpt.dh)

            s = np.where(m, -np.inf, s)

            p = softmax(s)

            ps.append(p); outs.append(p @ v[:, sl])

        co = np.concatenate(outs, -1)

        xm = x + co @ W['Wo']

        xr = gpt.rms(xm)

        u, zv = xr @ W['W1'], xr @ W['W3']

        gate = u/(1+np.exp(-u))*zv

        tape.append(dict(x_in=x, xm=xm, xr=xr, u=u, zv=zv, gate=gate, co=co, q=q, k=k, v=v, ps=ps))

        x = xm + gate @ W['W2']

    xf = x

    logits = gpt.rms(xf) @ gpt.head

    grads = {'E': np.zeros_like(gpt.E), 'head': np.zeros_like(gpt.head),

             'layers': [dict.fromkeys(W) for W in gpt.layers]}

    p = softmax(logits)

    onehot = np.zeros_like(p)

    onehot[np.arange(T), np.array(targets)] = 1.0

    dlogits = (p - onehot) * w[:, None]

    grads['head'] = gpt.rms(xf).T @ dlogits

    dx = rms_backward(dlogits @ gpt.head.T, xf)

    for i in reversed(range(gpt.NL)):

        W, t = gpt.layers[i], tape[i]

        dg = dx @ W['W2'].T

        du = dg * silu_grad(t['u']) * t['zv']

        dzv = dg * (t['u']/(1+np.exp(-t['u'])))

        grads['layers'][i]['W2'] = t['gate'].T @ dx

        grads['layers'][i]['W1'] = t['xr'].T @ du

        grads['layers'][i]['W3'] = t['xr'].T @ dzv

        dxm = dx + rms_backward(du @ W['W1'].T + dzv @ W['W3'].T, t['xm'])

        dc = dxm @ W['Wo'].T

        grads['layers'][i]['Wo'] = t['co'].T @ dxm

        dvs, dqs, dks = [], [], []

        for h in range(gpt.H):

            sl = slice(h*gpt.dh, (h+1)*gpt.dh)

            p_h, v_h, q_h, k_h = t['ps'][h], t['v'][:, sl], t['q'][:, sl], t['k'][:, sl]

            dO = dc[:, sl]

            dvs.append(p_h.T @ dO)

            dp = dO @ v_h.T

            ds = p_h * (dp - (dp*p_h).sum(-1, keepdims=True))

            dqs.append(ds @ k_h / math.sqrt(gpt.dh))

            dks.append(ds.T @ q_h / math.sqrt(gpt.dh))

        dv = np.concatenate(dvs, -1)

        dq = rope_backward(np.concatenate(dqs, -1), pos, gpt.D)

        dk = rope_backward(np.concatenate(dks, -1), pos, gpt.D)

        grads['layers'][i]['Wv'] = t['x_in'].T @ dv

        grads['layers'][i]['Wq'] = t['x_in'].T @ dq

        grads['layers'][i]['Wk'] = t['x_in'].T @ dk

        dx = dxm + dq @ W['Wq'].T + dk @ W['Wk'].T + dv @ W['Wv'].T

    np.add.at(grads['E'], np.array(ids), dx)

    return grads



def seq_logprob(m, ids, n_ans):

    """答案段(最后 n_ans+1 个 token:答案各字 + <|end|>)的 log 概率之和。"""

    logits, _ = m.forward(ids[:-1])

    tg = np.array(ids[1:])

    z = logits - logits.max(-1, keepdims=True)

    logp = z - np.log(np.exp(z).sum(-1, keepdims=True))

    lp = logp[np.arange(len(tg)), tg]

    return lp[-(n_ans+1):].sum()



def add_grads(a, b):

    return {'E': a['E'] + b['E'], 'head': a['head'] + b['head'],

            'layers': [{k: L1[k] + L2[k] for k in L1}

                       for L1, L2 in zip(a['layers'], b['layers'])]}



def greedy_answer(m, prompt_ids, max_new=5):

    ids = list(prompt_ids)

    for _ in range(max_new):

        logits, _ = m.forward(ids)

        nxt = int(np.argmax(logits[-1]))

        if nxt == T_END:

            break

        ids = ids + [nxt]

    return ids



def task_acc(m):

    ok = 0; total = 0

    for q, a in [(f"反义词:{x}", y) for x, y in ANTI] + [(f"重复:{s}", s) for s in COPY_STRINGS]:

        prompt = [T_USER] + enc(q) + [T_ASSI]

        got = greedy_answer(m, prompt, len(a) + 1)[len(prompt):]

        ok += ("".join(itos.get(i, '#') for i in got) == a); total += 1

    return ok, total

In [ ]:
# ===== GRPO 本体(讲义 §3) =====
import copy as _copy

def sample_answer(m, prompt, max_new, T_temp=1.0, rng=None):
    """从 m 采样一个回答(到 <|end|> 或 max_new);返回 (完整ids, 答案token列表)。"""
    rng = rng or np.random.default_rng()
    ids = list(prompt)
    ans = []
    for _ in range(max_new):
        logits, _ = m.forward(ids)          # 序列很短(≤12),整段重算,清晰第一
        p = softmax(logits[-1] / T_temp)
        nxt = int(rng.choice(len(p), p=p / p.sum()))
        if nxt == T_END:
            break
        ans.append(nxt)
        ids = ids + [nxt]
    return ids, ans

def answer_logprobs(m, prompt, ans):
    """回答各 token 的 logπ(list)。"""
    ids = list(prompt) + list(ans)
    logits, _ = m.forward(ids[:-1])
    z = logits - logits.max(-1, keepdims=True)
    logp = z - np.log(np.exp(z).sum(-1, keepdims=True))
    return [float(logp[len(prompt) - 1 + i, ans[i]]) for i in range(len(ans))]

def grpo_loss_and_grads(pol, ref, prompt, ans, old_lp, adv, beta_kl, clip_eps):
    """单个样本:返回 (loss, ws, lp_new)。w_t = (Â·c_t − β(e^{lr}−1))/n,标量 detach。"""
    lp_new = answer_logprobs(pol, prompt, ans)
    lp_ref = answer_logprobs(ref, prompt, ans)
    n = max(len(ans), 1)
    ratios = [math.exp(a_ - b_) for a_, b_ in zip(lp_new, old_lp)]
    un = [r * adv for r in ratios]
    clipped = [min(max(r, 1 - clip_eps), 1 + clip_eps) * adv for r in ratios]
    surr = sum(min(u, c) for u, c in zip(un, clipped)) / n
    kls = [math.exp(a_ - r_) - (a_ - r_) - 1 for a_, r_ in zip(lp_new, lp_ref)]
    loss = -(surr - beta_kl * sum(kls) / n)
    ws = []
    for r, u, c, lp_n, lp_r in zip(ratios, un, clipped, lp_new, lp_ref):
        coeff = r if u <= c else 0.0          # ← 被裁剪的分支是常数,梯度为 0
        ws.append((adv * coeff - beta_kl * (math.exp(lp_n - lp_r) - 1)) / n)
    return loss, ws, lp_new

In [ ]:
# ===== 验收:含未裁剪/越界/KL 非零三种情形(讲义 §4) =====
rng0 = np.random.default_rng(0)
m0 = MiniGPT(vocab=8, d=16, n_heads=2, n_layers=1, seed=3)
m0.E = np.vstack([m0.E, rng0.normal(size=(3, 16)) * 0.1])
m0.head = np.hstack([m0.head, rng0.normal(size=(16, 3)) * 0.1])
ref0 = _copy.deepcopy(m0)
ref0.E = ref0.E * 1.01                       # 参考与策略错开,KL 项非零
prompt = [3, 8, 5, 9]
ans = [4, 10]
old_lp = answer_logprobs(m0, prompt, ans)
old_lp = [old_lp[0] - 0.05, old_lp[1] - 0.5] # token0 在裁剪带内,token1 越界
adv, beta_kl, clip_eps = 0.7, 0.1, 0.2

def build():
    _, ws, _ = grpo_loss_and_grads(m0, ref0, prompt, ans, old_lp, adv, beta_kl, clip_eps)
    ids = prompt + ans
    w_full = [0.0] * (len(ids) - 1)
    for i, w in enumerate(ws):
        w_full[len(prompt) - 1 + i] = w
    return weighted_loss_and_grads(m0, ids[:-1], ids[1:], w_full)

g0 = build()
def f():
    return grpo_loss_and_grads(m0, ref0, prompt, ans, old_lp, adv, beta_kl, clip_eps)[0]
amap = [m0.E, m0.head] + [W for L in m0.layers for W in L.values()]
gmap = [g0['E'], g0['head']] + [W for L in g0['layers'] for W in L.values()]
errs = []
rr = np.random.default_rng(1)
for arr, gar in zip(amap, gmap):
    for _ in range(8):
        idx = tuple(rr.integers(0, s, 1)[0] for s in arr.shape)
        old = arr[idx]
        arr[idx] = old + 1e-6; fp = f()
        arr[idx] = old - 1e-6; fm = f()
        arr[idx] = old
        errs.append(abs(gar[idx] - (fp - fm) / 2e-6))
print(f"GRPO 梯度抽查 {len(errs)} 坐标 max|err| = {max(errs):.2e}")

In [ ]:
# ===== 基座 + 中等 SFT(讲义 §5) =====
t0 = time.time()
gpt = MiniGPT(vocab=V0, d=64, n_heads=4, n_layers=2, seed=0)
params = flat_params(gpt)
opt = MultiAdamW(params)
rng = np.random.default_rng(42)
T, B = 64, 16
for step in range(2000):
    opt.lr = lr_at(step, 2000)
    off = rng.integers(0, len(train_data) - T - 1, B)
    gsum = None
    for b in range(B):
        xb = train_data[off[b]:off[b] + T + 1]
        loss, grads = loss_and_grads(gpt, list(xb[:-1]), list(xb[1:]))
        gs = flat_grads(grads)
        if gsum is None:
            gsum = [g / B for g in gs]
        else:
            for a, g in zip(gsum, gs):
                a += g / B
    clip_norms(gsum, 1.0)
    opt.step(gsum)
extend_vocab(gpt)

def run_sft(m, steps, seed=0):
    ps_ = flat_params(m)
    o_ = MultiAdamW(ps_, lr=1e-4)
    rr_ = np.random.default_rng(seed)
    for step in range(steps):
        gsum = None
        idx = rr_.integers(0, len(ALL_TASKS), 16)
        for j in idx:
            ids_, mk_ = ALL_TASKS[j]
            loss, grads = loss_and_grads(m, ids_[:-1], ids_[1:], mk_[1:])
            gs = flat_grads(grads)
            if gsum is None:
                gsum = [g / 16 for g in gs]
            else:
                for a, g in zip(gsum, gs):
                    a += g / 16
        clip_norms(gsum, 1.0)
        o_.step(gsum)

def sample_success(m, T_temp=0.7, n_per=4, seed=9):
    rr_ = np.random.default_rng(seed)
    ok = tot = 0
    for q, truth in [(f"反义词:{x}", y) for x, y in ANTI] + [(f"重复:{s}", s) for s in COPY_STRINGS]:
        prompt = [T_USER] + enc(q) + [T_ASSI]
        for _ in range(n_per):
            _, ans = sample_answer(m, prompt, len(truth) + 1, T_temp=T_temp, rng=rr_)
            ok += ("".join(itos.get(i, '#') for i in ans) == truth)
            tot += 1
    return ok / tot

sft = _copy.deepcopy(gpt)
run_sft(sft, 400)
acc_sft = task_acc(sft)
f_sft, _ = val_loss_no_special(sft)
ss_sft = sample_success(sft)
print(f"[基座+中等SFT 400步]({time.time()-t0:.0f}s) 贪心 {acc_sft[0]}/50"
      f" | 采样成功率(T=0.7) {ss_sft:.0%} | val {f_sft:.4f}")

In [ ]:
# ===== GRPO 训练器 + 主实验 + 两组对照(讲义 §5–6) =====
QA = [(q, a, [T_USER] + enc(q) + [T_ASSI]) for q, a in
      ([(f"反义词:{x}", y) for x, y in ANTI] + [(f"重复:{s}", s) for s in COPY_STRINGS])]

def reward(ans_ids, truth):
    """答案对错(1.0)+ 格式(长度匹配 0.1)——R1 同款『答案+格式』奖励结构。"""
    s = "".join(itos.get(i, '#') for i in ans_ids)
    return (1.0 if s == truth else 0.0) + (0.1 if len(ans_ids) == len(truth) else 0.0)

def train_grpo(pol, ref_m, steps=300, G=8, n_q=4, lr=3e-5, beta_kl=0.05,
               clip_eps=0.2, epochs=1, seed=0, log_every=100, T_temp=0.7):
    rrr = np.random.default_rng(seed)
    ps2 = flat_params(pol)
    o2 = MultiAdamW(ps2, lr=lr)
    ref_cache = {}
    reward_curve, degen_count, total_groups = [], 0, 0
    for step in range(steps):
        qidx = rrr.integers(0, len(QA), n_q)
        batch = []
        rs_mean_step = []
        for k in qidx:
            q, truth, prompt = QA[k]
            group = []
            for g in range(G):
                _, ans = sample_answer(pol, prompt, len(truth) + 1, T_temp=T_temp, rng=rrr)
                if not ans:
                    ans = [int(rrr.integers(0, V0))]
                r = reward(ans, truth)
                group.append((ans, answer_logprobs(pol, prompt, ans), r))
            rs = np.array([g_[2] for g_ in group])
            rs_mean_step.append(rs.mean())
            total_groups += 1
            std = rs.std()
            if std < 1e-6:
                degen_count += 1          # 全对或全错:没有梯度
                continue
            for ans, lp, r in group:
                batch.append((prompt, ans, lp, (r - rs.mean()) / (std + 1e-4)))
        reward_curve.append(np.mean(rs_mean_step))
        if not batch:
            continue
        for ep in range(epochs):
            gsum = None
            for prompt, ans, old_lp, adv in batch:
                key = (tuple(prompt), tuple(ans))
                if key not in ref_cache:
                    ref_cache[key] = answer_logprobs(ref_m, prompt, ans)
                loss, ws, _ = grpo_loss_and_grads(pol, ref_m, prompt, ans,
                                                  old_lp, adv, beta_kl, clip_eps)
                w_full = [0.0] * (len(prompt) + len(ans) - 1)
                for i, w in enumerate(ws):
                    w_full[len(prompt) - 1 + i] = w
                grads = weighted_loss_and_grads(pol, prompt + ans[:-1],
                                                prompt[1:] + ans, w_full)
                gs = flat_grads(grads)
                if gsum is None:
                    gsum = [x / len(batch) for x in gs]
                else:
                    for a2, x in zip(gsum, gs):
                        a2 += x / len(batch)
            clip_norms(gsum, 1.0)
            o2.step(gsum)
        if (step + 1) % log_every == 0:
            print(f"    step {step+1}: 组均奖励 {reward_curve[-1]:.2f}"
                  f" | 贪心 {task_acc(pol)[0]}/50 | val {val_loss_no_special(pol)[0]:.4f}")
    return np.array(reward_curve), degen_count, total_groups

print("[GRPO 主实验] G=8 β_kl=0.05 ε=0.2 lr=3e-5 300 步:")
pol = _copy.deepcopy(sft)
ref = _copy.deepcopy(sft)
t1 = time.time()
rc, degen, tot_g = train_grpo(pol, ref)
print(f"  用时 {time.time()-t1:.0f}s")
acc_g = task_acc(pol)
f_g, _ = val_loss_no_special(pol)
ss_g = sample_success(pol)
print(f"  组均奖励:步0 {rc[0]:.2f} → 步100 {rc[99]:.2f} → 步200 {rc[199]:.2f} → 步300 {rc[-1]:.2f}")
print(f"  贪心 {acc_sft[0]}/50 → {acc_g[0]}/50 | 采样成功率 {ss_sft:.0%} → {ss_g:.0%}"
      f" | val {f_sft:.4f} → {f_g:.4f}")
print(f"  退化组:{degen}/{tot_g} = {degen/tot_g:.0%}")

print("\n[对照A] β_kl=0,其余同:")
pol2 = _copy.deepcopy(sft)
ref2 = _copy.deepcopy(sft)
rc2, _, _ = train_grpo(pol2, ref2, beta_kl=0.0, log_every=300)
print(f"  贪心 {task_acc(pol2)[0]}/50 | 采样成功率 {sample_success(pol2):.0%}"
      f" | val {val_loss_no_special(pol2)[0]:.4f} | 组均奖励 {rc2[0]:.2f} → {rc2[-1]:.2f}")

print("\n[对照B] 弱起点(SFT 150 步),同药方:")
sft_weak = _copy.deepcopy(gpt)
run_sft(sft_weak, 150)
pol3 = _copy.deepcopy(sft_weak)
ref3 = _copy.deepcopy(sft_weak)
rc3, dg3, tg3 = train_grpo(pol3, ref3, log_every=300)
print(f"  贪心 {task_acc(sft_weak)[0]}/50 → {task_acc(pol3)[0]}/50"
      f" | 组均奖励 {rc3[0]:.2f} → {rc3[-1]:.2f} | val {val_loss_no_special(pol3)[0]:.4f}"
      f" | 退化组 {dg3/tg3:.0%}")

print("\n[训练篇终章] 四段旅程:")
print(f"  base(24课):贪心 0/50,只会续写")
print(f"  SFT 400步:贪心 {acc_sft[0]}/50,采样成功率 {ss_sft:.0%}")
print(f"  DPO(27课,弱SFT起点):Δ 100% 拉开,贪心 18→16——偏好分离,不涨点")
print(f"  GRPO(本课):贪心 {acc_g[0]}/50,采样成功率 {ss_sft:.0%}→{ss_g:.0%}——巩固已有能力")

In [ ]:
# ---- 作业现场 ----

# TODO(C1): 默写 grpo_loss_and_grads(裁剪零梯度 + KL 负号),过数值裁判

# TODO(C2): G∈{2,4,16} 的退化组率统计(固定策略,只采样)

# TODO(C3): k3 = e^lr − lr − 1:验证 lr=0 处最小值 0、恒非负

# TODO(D1): 7B GRPO vs PPO 显存账

# TODO(D3): 查证 R1 四阶段 / R1-Zero

# TODO(E2/预习): prefill 与 decode 的两副脾气分别是什么?
